# Part 1: Exploratory Data Analysis

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import re
import html
from collections import Counter

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
from wordcloud import WordCloud, STOPWORDS

pio.templates.default = 'plotly_white'
path = '/content/drive/MyDrive/project2/'

## Data

<div dir="rtl">

بعضی asinها در جدول محصولات دو بار آمده‌اند و merge را بزرگ می‌کنند. این تکراری‌ها و نظرهای تکراری (همان نویسنده، همان محصول، همان متن) حذف شدند.

</div>

In [ ]:
train_df = pd.read_csv(path + 'train_data.csv', low_memory=False)
title_brand_df = pd.read_csv(path + 'title_brand.csv')
print(train_df.shape, title_brand_df.shape)

In [ ]:
key = ['reviewerID', 'asin', 'reviewText']
print('duplicate reviews:', train_df.duplicated(key).sum())
print('duplicate asins:', title_brand_df['asin'].duplicated().sum())

train_df = train_df.drop_duplicates(key)
title_brand_df = title_brand_df.drop_duplicates('asin')
title_brand_df['title'] = title_brand_df['title'].map(html.unescape, na_action='ignore')
print(train_df.shape, title_brand_df.shape)

## Q1: Rating distribution

In [ ]:
counts = train_df['overall'].value_counts().sort_index()
rating_summary = pd.DataFrame({'rating': counts.index.astype(str),
                               'count': counts.values,
                               'percent': (counts.values / counts.sum() * 100).round(2)})
display(rating_summary)

rating_colors = {'1': '#B71C1C', '2': '#E53935', '3': '#616161', '4': '#66BB6A', '5': '#1B5E20'}
fig = px.bar(rating_summary, x='rating', y='count', color='rating', text='count', custom_data=['percent'],
             color_discrete_map=rating_colors, title='Rating distribution')
fig.update_traces(textposition='outside', hovertemplate='rating %{x}<br>%{y:,} reviews<br>%{customdata[0]:.2f}%<extra></extra>')
fig.update_layout(showlegend=False, xaxis_type='category')
fig.show()

In [ ]:
sentiment = train_df['overall'].map({1: 'negative', 2: 'negative', 3: 'neutral', 4: 'positive', 5: 'positive'})
counts = sentiment.value_counts().reindex(['negative', 'neutral', 'positive'])
sentiment_summary = pd.DataFrame({'sentiment': counts.index,
                                  'count': counts.values,
                                  'percent': (counts.values / counts.sum() * 100).round(2)})
display(sentiment_summary)

sentiment_colors = {'negative': '#D32F2F', 'neutral': '#616161', 'positive': '#2E7D32'}
fig = px.bar(sentiment_summary, x='sentiment', y='count', color='sentiment', text='count', custom_data=['percent'],
             color_discrete_map=sentiment_colors, title='Sentiment classes')
fig.update_traces(textposition='outside', hovertemplate='%{x}<br>%{y:,} reviews<br>%{customdata[0]:.2f}%<extra></extra>')
fig.update_layout(showlegend=False)
fig.show()

<div dir="rtl">

داده نامتوازن است: امتیاز ۵ به‌تنهایی ۵۴.۹۵٪ نظرهاست و کلاس مثبت ۷۳.۶۳٪.

معیار ارزیابی پروژه micro-F1 است که در طبقه‌بندی تک‌برچسبی همان accuracy است. متوازن کردن داده (مثلاً `class_weight='balanced'` یا oversampling کلاس‌های کوچک) recall کلاس‌های ۲ و ۳ را بالا می‌برد ولی معمولاً accuracy کل را پایین می‌آورد. پیشنهاد ما این است که در مدل‌سازی هر دو حالت روی داده‌ی validation مقایسه شوند و حالتی انتخاب شود که micro-F1 بهتری بدهد.

</div>

## Q2: Word clouds

<div dir="rtl">

فراوانی کلمات نظر به نظر شمرده شد تا کل متن‌ها یک‌جا در حافظه نباشند. کد HTML حذف شد و آپاستروف نگه داشته شد تا don't و didn't تکه‌تکه نشوند و همراه stop wordها کنار بروند.

</div>

In [ ]:
stopwords = set(STOPWORDS) | {'amazon', 'product', 'buy', 'bought', 'review', 'purchase', 'would', 'could',
                              'really', 'also', 'one', 'get', 'use', 'used', 'like', 'just', 'much', 'well',
                              'even', 'still', 'thing', 'things', 'time', 'work', 'works', 'make', 'need',
                              'will', 'now', 'using', 'got', 'two', 'back', 'first', 'new', 'see', 'way', 'set',
                              'want', 'right', 'know', 'going', 'something', 'think', 'around', 'sure', 'many',
                              'able', 'put', 'take', 'look', 'say', 'said'}

def word_counts(reviews):
    counts = Counter()
    for text in reviews:
        text = re.sub(r'<[^>]+>', ' ', text.lower())
        words = [w.removesuffix("'s") for w in re.findall(r"[a-z]+(?:'[a-z]+)?", text)]
        counts.update(w for w in words if len(w) > 2 and w not in stopwords)
    return counts

groups = {'positive': train_df.loc[train_df['overall'] >= 4, 'reviewText'],
          'neutral': train_df.loc[train_df['overall'] == 3, 'reviewText'],
          'negative': train_df.loc[train_df['overall'] <= 2, 'reviewText']}
freqs = {name: word_counts(reviews) for name, reviews in groups.items()}

In [ ]:
colormaps = {'positive': 'Greens', 'neutral': 'Greys', 'negative': 'Reds'}

for name, freq in freqs.items():
    cloud = WordCloud(width=800, height=400, background_color='white', max_words=100,
                      colormap=colormaps[name], random_state=42).generate_from_frequencies(freq)
    fig = px.imshow(cloud.to_array(), title=f'{name.capitalize()} reviews')
    fig.update_xaxes(visible=False)
    fig.update_yaxes(visible=False)
    fig.show()

In [ ]:
top_positive = {w for w, _ in freqs['positive'].most_common(50)}
top_negative = {w for w, _ in freqs['negative'].most_common(50)}

print('common:', len(top_positive & top_negative), sorted(top_positive & top_negative))
print('only positive:', sorted(top_positive - top_negative))
print('only negative:', sorted(top_negative - top_positive))

<div dir="rtl">

از ۵۰ کلمه‌ی پرتکرار نظرهای مثبت و منفی، ۳۰ کلمه مشترک‌اند. بیشترشان اسم موضوع یا قطعه‌اند (مثل battery، camera، sound، screen و quality). یعنی هر دو گروه درباره‌ی چیزهای یکسانی حرف می‌زنند و تفاوت در قضاوت‌شان است، نه موضوع. کلمه‌هایی که فقط در یک گروه پرتکرارند همین را نشان می‌دهند: در نظرهای مثبت love، easy، nice، perfect و recommend و در نظرهای منفی return، problem، issue، never، money و support.

STOPWORDS کلمه‌هایی مثل not و no را هم حذف می‌کند. برای ابر کلمات این مشکلی ندارد، ولی در مدل‌سازی این کلمه‌ها نگه داشته می‌شوند چون معنی جمله را برعکس می‌کنند.

</div>

## Q3: Most helpful reviewers

<div dir="rtl">

ستون vote متن است و رأی‌های بالای هزار با کاما نوشته شده‌اند (مثل "1,489")؛ قبل از تبدیل به عدد کاما حذف شد.

</div>

In [ ]:
votes = pd.to_numeric(train_df['vote'].astype(str).str.replace(',', ''), errors='coerce').fillna(0)

top_reviewers = (train_df.assign(vote=votes)
                 .groupby('reviewerID')
                 .agg(name=('reviewerName', 'first'), total_votes=('vote', 'sum'))
                 .nlargest(10, 'total_votes'))
top_reviewers['total_votes'] = top_reviewers['total_votes'].astype(int)
display(top_reviewers)

fig = px.bar(top_reviewers.sort_values('total_votes'), x='total_votes', y='name', orientation='h',
             text='total_votes', title='10 reviewers with the most helpful votes')
fig.update_traces(textposition='outside')
fig.show()

<div dir="rtl">

Benson Leung با ۴٬۵۷۲ رأی اول است و بعد از او Voltron00x (۳٬۱۸۱) و Obi Wan (۳٬۱۵۰) قرار دارند. گروه‌بندی بر اساس reviewerID انجام شد چون اسم‌ها یکتا نیستند.

</div>

## Q4: Review length

<div dir="rtl">

طول متن بعد از حذف کد HTML حساب شد؛ حدود ۱۷ هزار نظر لینک یا ویدئوی HTML دارند و طولشان بیشتر از واقعی دیده می‌شد.

</div>

In [ ]:
lengths = train_df['reviewText'].str.replace(r'<[^>]+>', ' ', regex=True).str.strip().str.len()
print('shortest raw text:', train_df['reviewText'].str.len().min())
print('shortest clean text:', lengths.min())
print('clean texts under 200 characters:', (lengths < 200).sum())
print('empty after cleaning:', (lengths == 0).sum())

In [ ]:
fig = px.histogram(lengths, nbins=60, title='Review length (characters)')
fig.update_layout(showlegend=False, xaxis_title='characters', yaxis_title='reviews')
fig.show()

In [ ]:
q1, q3 = lengths.quantile([0.25, 0.75])
upper = q3 + 1.5 * (q3 - q1)
filtered = lengths[lengths <= upper]
print('Q1:', q1, ' Q3:', q3, ' upper bound:', upper)
print('kept:', len(filtered), f'({len(filtered) / len(lengths):.1%})', ' outliers:', len(lengths) - len(filtered))

fig = px.histogram(filtered, nbins=50, title=f'Review length without outliers (<= {upper:.0f} characters)')
fig.update_layout(showlegend=False, xaxis_title='characters', yaxis_title='reviews')
fig.show()

<div dir="rtl">

کوتاه‌ترین متن خام دقیقاً ۲۰۰ کاراکتر است؛ یعنی داده از قبل فیلتر شده و هیستوگرام خام از ۲۰۰ شروع می‌شود. بعد از حذف HTML، ۱٬۳۴۲ نظر کوتاه‌تر از ۲۰۰ کاراکترند و ۵ نظر کاملاً خالی می‌شوند (فقط ویدئو یا لینک داشته‌اند). با روش IQR کران پایین منفی می‌شود، پس پرت‌ها فقط نظرهای خیلی بلندند (بیشتر از ۱۳۱۴ کاراکتر، حدود ۸.۵٪ نظرها).

برای مدل‌هایی مثل TF-IDF محدودیت طول لازم نیست. برای مدل‌هایی که ورودی با طول ثابت می‌گیرند، ۱۳۱۴ کاراکتر نقطه‌ی شروع خوبی برای حداکثر طول است: ۹۱.۵٪ نظرها کامل جا می‌شوند و نظرهای بلندتر کوتاه می‌شوند، نه حذف، چون اطلاعاتشان به کار می‌آید.

</div>

## Q5: Products with the most 5-star ratings

In [ ]:
five_star = (train_df[train_df['overall'] == 5]
             .groupby('asin').size().rename('five_star_reviews')
             .nlargest(10).reset_index())
top_products = five_star.merge(title_brand_df, on='asin', how='left')
top_products[['brand', 'title', 'five_star_reviews']]

<div dir="rtl">

بلندگوی OontZ Angle 3 (برند Cambridge Soundworks) با ۱٬۱۶۳ نظر ۵ ستاره در صدر است. قبل از حذف نظرهای تکراری دو محصول دیگر (StarTech USB2SATAIDE و VideoSecu ML531BE) هم در این فهرست بودند؛ تقریباً همه‌ی نظرهای این دو محصول دو بار در داده آمده‌اند.

</div>

## Q6: Average rating of the 10 most reviewed brands

In [ ]:
reviews = train_df.merge(title_brand_df[['asin', 'brand']], on='asin', how='left')
top_brands = (reviews.groupby('brand')['overall']
              .agg(reviews='size', mean_rating='mean')
              .nlargest(10, 'reviews')
              .sort_values('mean_rating', ascending=False)
              .round(2))
top_brands

<div dir="rtl">

Anker با میانگین ۴.۲۴ اول و AmazonBasics با ۴.۲۰ دوم است؛ NETGEAR با ۳.۸۱ کمترین میانگین را بین این ده برند دارد. فاصله‌ی بیشترین و کمترین میانگین کمتر از نیم ستاره است.

</div>